# 07 — Gemma 4 Embedding Analysis

Interactive exploration of the multimodal embeddings generated by `google/gemma-4-E4B-it`.

**Prerequisites:** run `python embeddings/generate_embeddings.py` first.

---

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.preprocessing import LabelEncoder

ROOT = Path("..").resolve()
EMB_PATH = ROOT / "embeddings" / "creative_embeddings.npz"

STATUS_COLORS = {
    "top_performer": "#2ecc71",
    "stable": "#3498db",
    "fatigued": "#e67e22",
    "underperformer": "#e74c3c",
}
print("Ready")

## 1. Load & Merge

In [ ]:
assert EMB_PATH.exists(), f"Run generate_embeddings.py first — {EMB_PATH} not found"

data = np.load(EMB_PATH)
X = data["embeddings"]  # (N, hidden_dim), already L2-normalised
creative_ids = data["creative_ids"]

df = pd.read_csv(ROOT / "creative_summary.csv")
emb_df = pd.DataFrame({"creative_id": creative_ids.astype(int)}).merge(
    df, on="creative_id", how="left"
)

print(f"Embeddings shape : {X.shape}")
print(f"Metadata rows    : {len(emb_df)}")
print(f"Status counts\n{emb_df['creative_status'].value_counts()}")

## 2. PCA

In [ ]:
pca = PCA(n_components=3, random_state=42)
coords = pca.fit_transform(X)
emb_df[["pc1", "pc2", "pc3"]] = coords

ev = pca.explained_variance_ratio_ * 100
print(f"Explained variance: PC1={ev[0]:.1f}%  PC2={ev[1]:.1f}%  PC3={ev[2]:.1f}%")

fig = px.scatter(
    emb_df,
    x="pc1",
    y="pc2",
    color="creative_status",
    color_discrete_map=STATUS_COLORS,
    hover_data=["creative_id", "headline", "vertical", "perf_score"],
    title=f"PCA 2D — coloured by status (PC1={ev[0]:.1f}%, PC2={ev[1]:.1f}%)",
    opacity=0.7,
)
fig.update_traces(marker=dict(size=5))
fig.show()

In [ ]:
fig3d = px.scatter_3d(
    emb_df,
    x="pc1",
    y="pc2",
    z="pc3",
    color="creative_status",
    color_discrete_map=STATUS_COLORS,
    hover_data=["creative_id", "headline", "vertical", "perf_score"],
    title="PCA 3D",
    opacity=0.7,
    size_max=5,
)
fig3d.update_traces(marker=dict(size=3))
fig3d.show()

In [ ]:
fig = px.scatter(
    emb_df,
    x="pc1",
    y="pc2",
    color="perf_score",
    color_continuous_scale="RdYlGn",
    hover_data=["creative_id", "headline", "creative_status"],
    title="PCA 2D — coloured by perf_score (green = best)",
    opacity=0.75,
)
fig.update_traces(marker=dict(size=5))
fig.show()

## 3. UMAP

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
umap_2d = reducer.fit_transform(X)
emb_df[["u1", "u2"]] = umap_2d

for color_col, title in [
    ("creative_status", "UMAP — by status"),
    ("vertical", "UMAP — by vertical"),
    ("theme", "UMAP — by theme"),
]:
    color_map = STATUS_COLORS if color_col == "creative_status" else None
    fig = px.scatter(
        emb_df,
        x="u1",
        y="u2",
        color=color_col,
        color_discrete_map=color_map,
        hover_data=["creative_id", "headline", "perf_score", "creative_status"],
        title=title,
        opacity=0.7,
    )
    fig.update_traces(marker=dict(size=5))
    fig.show()

## 4. KMeans Clustering

In [ ]:
ks = [2, 4, 6, 8, 10]
sil_scores = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels, sample_size=min(500, len(X)))
    sil_scores.append(score)
    print(f"  k={k}  silhouette={score:.4f}")

plt.figure(figsize=(7, 3))
plt.plot(ks, sil_scores, "o-")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Scores for KMeans")
plt.tight_layout()
plt.show()

best_k = ks[int(np.argmax(sil_scores))]
print(f"Best k = {best_k}")

In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
emb_df["cluster"] = km.fit_predict(X).astype(str)

fig = px.scatter(
    emb_df,
    x="u1",
    y="u2",
    color="cluster",
    symbol="creative_status",
    hover_data=["creative_id", "headline", "vertical", "perf_score"],
    title=f"UMAP — KMeans clusters (k={best_k})",
    opacity=0.75,
)
fig.update_traces(marker=dict(size=6))
fig.show()

In [ ]:
profiles = (
    emb_df.groupby("cluster")
    .agg(
        count=("creative_id", "count"),
        avg_perf=("perf_score", "mean"),
        top_pct=("creative_status", lambda x: (x == "top_performer").mean()),
        under_pct=("creative_status", lambda x: (x == "underperformer").mean()),
        top_vertical=("vertical", lambda x: x.value_counts().index[0]),
        top_theme=("theme", lambda x: x.value_counts().index[0]),
        top_hook=("hook_type", lambda x: x.value_counts().index[0]),
    )
    .round(3)
)
profiles.columns = ["N", "Avg perf", "TopPerf%", "Under%", "Top vertical", "Top theme", "Top hook"]
print(profiles.to_string())

## 5. DBSCAN (Density-Based Clusters)

In [ ]:
# Use PCA-reduced embeddings for DBSCAN to reduce noise
pca50 = PCA(n_components=50, random_state=42)
X50 = pca50.fit_transform(X)

db = DBSCAN(eps=0.5, min_samples=5, metric="euclidean", n_jobs=-1)
db_labels = db.fit_predict(X50)

emb_df["dbscan_cluster"] = db_labels.astype(str)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()
print(f"DBSCAN: {n_clusters} clusters, {n_noise} noise points")

fig = px.scatter(
    emb_df,
    x="u1",
    y="u2",
    color="dbscan_cluster",
    hover_data=["creative_id", "headline", "creative_status", "perf_score"],
    title=f"UMAP — DBSCAN clusters ({n_clusters} clusters, {n_noise} noise)",
    opacity=0.75,
)
fig.update_traces(marker=dict(size=5))
fig.show()

## 6. Cosine Similarity — Top Performers

In [ ]:
top_mask = emb_df["creative_status"] == "top_performer"
top_X = X[top_mask.values]
top_ids = emb_df[top_mask]["creative_id"].astype(str).tolist()

sim = top_X @ top_X.T  # dot product of L2-normalised vectors = cosine similarity

n_show = min(25, len(top_X))
fig = px.imshow(
    sim[:n_show, :n_show],
    x=top_ids[:n_show],
    y=top_ids[:n_show],
    color_continuous_scale="Viridis",
    zmin=0.5,
    zmax=1.0,
    title=f"Cosine Similarity — Top {n_show} Top-Performers",
    aspect="equal",
)
fig.update_layout(height=520)
fig.show()

# Find most similar pairs
np.fill_diagonal(sim, -1)
pairs = []
for i in range(len(top_X)):
    j = int(np.argmax(sim[i]))
    if i < j:
        pairs.append((top_ids[i], top_ids[j], float(sim[i, j])))
pairs.sort(key=lambda x: -x[2])
print("Most similar top-performer pairs:")
for a, b, s in pairs[:10]:
    print(f"  {a} ↔ {b}  cosine={s:.4f}")

## 7. Linear Probing — How much performance signal is in the embeddings?

In [ ]:
le = LabelEncoder()
y = le.fit_transform(emb_df["creative_status"].fillna("unknown"))

n_train = int(0.8 * len(X))
X_tr, X_te = X[:n_train], X[n_train:]
y_tr, y_te = y[:n_train], y[n_train:]

clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
clf.fit(X_tr, y_tr)
preds = clf.predict(X_te)

acc = accuracy_score(y_te, preds)
print(f"Linear probing accuracy: {acc:.2%}  (random baseline: 25%)")
print()
print(classification_report(y_te, preds, target_names=le.classes_))

## 8. Text Embeddings — Joint Multimodal Space

Embed `headline + cta_text` using the same Gemma 4 model (text-only) and overlay with image embeddings in shared PCA space.

In [ ]:
TEXT_EMB_PATH = ROOT / "embeddings" / "text_embeddings.npz"

if TEXT_EMB_PATH.exists():
    text_data = np.load(TEXT_EMB_PATH)
    X_text = text_data["embeddings"]
    text_ids = text_data["creative_ids"]

    # Joint PCA
    all_X = np.vstack([X, X_text])
    pca_joint = PCA(n_components=2, random_state=42)
    all_coords = pca_joint.fit_transform(all_X)

    n = len(X)
    joint_df = pd.concat(
        [
            pd.DataFrame(
                {
                    "pc1": all_coords[:n, 0],
                    "pc2": all_coords[:n, 1],
                    "modality": "image",
                    "creative_id": creative_ids.astype(int),
                }
            ),
            pd.DataFrame(
                {
                    "pc1": all_coords[n:, 0],
                    "pc2": all_coords[n:, 1],
                    "modality": "text",
                    "creative_id": text_ids.astype(int),
                }
            ),
        ]
    ).merge(df[["creative_id", "creative_status", "vertical"]], on="creative_id", how="left")

    fig = px.scatter(
        joint_df,
        x="pc1",
        y="pc2",
        color="creative_status",
        symbol="modality",
        color_discrete_map=STATUS_COLORS,
        hover_data=["creative_id", "vertical"],
        title="Joint PCA: image embeddings (●) vs text embeddings (▲)",
        opacity=0.7,
    )
    fig.update_traces(marker=dict(size=5))
    fig.show()
else:
    print("Text embeddings not found. To generate:")
    print("  python embeddings/generate_text_embeddings.py")

## 9. Nearest-Neighbour Recommendations

Given a fatigued creative, find the most visually similar top-performer to replace it.

In [ ]:
fatigued_mask = emb_df["creative_status"] == "fatigued"
top_mask = emb_df["creative_status"] == "top_performer"

X_fatigued = X[fatigued_mask.values]
X_top = X[top_mask.values]

fat_ids = emb_df[fatigued_mask]["creative_id"].tolist()
top_ids_all = emb_df[top_mask][["creative_id", "headline", "vertical", "perf_score"]].reset_index(
    drop=True
)

# For each fatigued creative, find most similar top-performer
sim_cross = X_fatigued @ X_top.T  # (n_fatigued, n_top)
best_match_idx = sim_cross.argmax(axis=1)

recs = []
for i, fat_id in enumerate(fat_ids[:10]):
    match = top_ids_all.iloc[best_match_idx[i]]
    recs.append(
        {
            "fatigued_id": fat_id,
            "recommended_id": match["creative_id"],
            "rec_headline": match["headline"],
            "rec_vertical": match["vertical"],
            "rec_perf": round(match["perf_score"], 3),
            "cosine_sim": round(float(sim_cross[i, best_match_idx[i]]), 4),
        }
    )

recs_df = pd.DataFrame(recs)
print("Nearest top-performer replacements for fatigued creatives:")
print(recs_df.to_string(index=False))